In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/indian_roads_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)

Original shape: (20000, 24)


In [2]:
clean_df = df.copy()

print("Working copy created.")
print("Shape:", clean_df.shape)

Working copy created.
Shape: (20000, 24)


In [3]:
clean_df = clean_df.drop(columns=["festival"])

print("festival column removed.")
print("New shape:", clean_df.shape)

festival column removed.
New shape: (20000, 23)


In [4]:
clean_df["date"] = pd.to_datetime(
    clean_df["date"],
    errors="coerce"
)

clean_df["time"] = pd.to_datetime(
    clean_df["time"],
    format="%H:%M",
    errors="coerce"
).dt.time
print(clean_df[["date", "time"]].dtypes)

date    datetime64[us]
time            object
dtype: object


In [5]:
missing = clean_df.isnull().sum()

print("========== MISSING VALUES AFTER CLEANING ==========")
print(missing[missing > 0])

========== MISSING VALUES AFTER CLEANING ==========
Series([], dtype: int64)


In [6]:
print("========== COORDINATE VALIDATION ==========")

invalid_lat = (
    (clean_df["latitude"] < -90) |
    (clean_df["latitude"] > 90)
).sum()

invalid_lon = (
    (clean_df["longitude"] < -180) |
    (clean_df["longitude"] > 180)
).sum()

print("Invalid latitude:", invalid_lat)
print("Invalid longitude:", invalid_lon)

========== COORDINATE VALIDATION ==========
Invalid latitude: 0
Invalid longitude: 0


In [7]:
print("Duplicate complete rows:", clean_df.duplicated().sum())
print(
    "Duplicate accident IDs:",
    clean_df["accident_id"].duplicated().sum()
)

Duplicate complete rows: 0
Duplicate accident IDs: 0


In [8]:
OUTPUT_PATH = "../data/processed/cleaned_accidents.csv"

clean_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Cleaned dataset saved successfully!")
print("Path:", OUTPUT_PATH)
print("Final shape:", clean_df.shape)

Cleaned dataset saved successfully!
Path: ../data/processed/cleaned_accidents.csv
Final shape: (20000, 23)


high-risk location + time combinations

In [9]:
print("========== TIME WINDOW PREPARATION ==========")

# Create 2-hour time windows
clean_df["time_window"] = (clean_df["hour"] // 2) * 2

# Convert to readable format
clean_df["time_window_label"] = (
    clean_df["time_window"].astype(str).str.zfill(2)
    + ":00 - "
    + (clean_df["time_window"] + 1).astype(str).str.zfill(2)
    + ":59"
)

print("Unique time windows:")
print(sorted(clean_df["time_window"].unique()))

print("\nTime-window distribution:")
print(clean_df["time_window_label"].value_counts().sort_index())

========== TIME WINDOW PREPARATION ==========
Unique time windows:
[np.int64(0), np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(10), np.int64(12), np.int64(14), np.int64(16), np.int64(18), np.int64(20), np.int64(22)]

Time-window distribution:
time_window_label
00:00 - 01:59    1699
02:00 - 03:59    1712
04:00 - 05:59    1632
06:00 - 07:59    1642
08:00 - 09:59    1635
10:00 - 11:59    1630
12:00 - 13:59    1744
14:00 - 15:59    1649
16:00 - 17:59    1648
18:00 - 19:59    1663
20:00 - 21:59    1668
22:00 - 23:59    1678
Name: count, dtype: int64


In [10]:
print("========== COORDINATE PRECISION ==========")

print("Unique latitude:", clean_df["latitude"].nunique())
print("Unique longitude:", clean_df["longitude"].nunique())
print(
    "Unique exact coordinate pairs:",
    clean_df[["latitude", "longitude"]].drop_duplicates().shape[0]
)

print("\nSample coordinates:")
display(clean_df[["city", "latitude", "longitude"]].head(10))

========== COORDINATE PRECISION ==========
Unique latitude: 19907
Unique longitude: 19927
Unique exact coordinate pairs: 20000

Sample coordinates:


,city,latitude,longitude
0,Pune,18.680827,73.930388
1,Mumbai,18.817732,72.790846
2,Mumbai,19.096889,72.819424
3,Chandigarh,30.787805,76.847507
4,Chennai,12.965155,80.283313
5,Delhi,28.799490,77.049666
6,Bangalore,13.064327,77.530941
7,Chandigarh,30.786617,76.733947
8,Hyderabad,17.422447,78.464881
9,Bangalore,12.939033,77.498966


In [11]:
print("========== SPATIAL GRID INSPECTION ==========")

GRID_SIZE = 0.01

clean_df["lat_grid"] = (
    np.floor(clean_df["latitude"] / GRID_SIZE) * GRID_SIZE
).round(2)

clean_df["lon_grid"] = (
    np.floor(clean_df["longitude"] / GRID_SIZE) * GRID_SIZE
).round(2)

grid_counts = (
    clean_df
    .groupby(["city", "lat_grid", "lon_grid"])
    .size()
    .reset_index(name="accident_count")
    .sort_values("accident_count", ascending=False)
)

print("Number of occupied grid cells:", len(grid_counts))

print("\nTop 20 grid cells:")
display(grid_counts.head(20))

========== SPATIAL GRID INSPECTION ==========
Number of occupied grid cells: 8105

Top 20 grid cells:


,city,lat_grid,lon_grid,accident_count
1585,Chandigarh,30.76,76.88,14
1389,Chandigarh,30.67,76.72,14
1298,Chandigarh,30.62,76.80,14
1541,Chandigarh,30.74,76.84,14
1268,Chandigarh,30.61,76.70,13
1369,Chandigarh,30.66,76.72,13
1274,Chandigarh,30.61,76.76,13
1349,Chandigarh,30.65,76.72,13
1293,Chandigarh,30.62,76.75,13
1556,Chandigarh,30.75,76.79,12


In [12]:
print("========== LOCATION + TIME DISTRIBUTION ==========")

location_time = (
    clean_df
    .groupby(
        ["city", "lat_grid", "lon_grid", "time_window"]
    )
    .size()
    .reset_index(name="accident_count")
)

print("Total location-time combinations:", len(location_time))

print("\nAccidents per location-time combination:")
display(location_time["accident_count"].describe())

========== LOCATION + TIME DISTRIBUTION ==========
Total location-time combinations: 17988

Accidents per location-time combination:


count    17988.000000
mean         1.111852
std          0.351865
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          7.000000
Name: accident_count, dtype: float64

In [13]:
print("\nTop 20 location-time combinations:")
display(
    location_time
    .sort_values("accident_count", ascending=False)
    .head(20)
)


Top 20 location-time combinations:


,city,lat_grid,lon_grid,time_window,accident_count
3003,Chandigarh,30.67,76.72,4,7
2409,Chandigarh,30.61,76.76,10,4
3256,Chandigarh,30.69,76.83,10,4
2904,Chandigarh,30.66,76.72,6,4
4259,Chandigarh,30.79,76.87,22,4
2510,Chandigarh,30.62,76.75,4,4
5661,Chennai,13.05,80.10,22,4
3998,Chandigarh,30.77,76.71,22,4
2408,Chandigarh,30.61,76.76,6,4
4328,Chennai,12.80,80.29,8,4
